# Variational Quantum Circuit (VQC) — Phishing URL Detection

Hybrid quantum-classical binary classifier. The data pipeline (TF-IDF → SVD → StandardScaler → PCA → MinMaxScaler) is identical to all other models in this project for a fair comparison.

**Circuit:** 4 qubits, 3 variational layers, angle encoding, CNOT ring entanglement.  
**Gradient method:** parameter-shift rule via PennyLane autograd interface (no PyTorch required).  
**Optimizer:** Adam (implemented manually over `pennylane.numpy` arrays).  
**Trainable params:** 24 VQC rotation angles + 4 classical weights + 1 bias = **29 total**.

## 1. Imports & CONFIG

In [1]:
import os
import re
import random

import numpy as np
import pennylane as qml
import pennylane.numpy as pnp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, PCA
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

np.random.seed(42)
random.seed(42)

print(f"PennyLane version : {qml.__version__}")

PennyLane version : 0.45.0


In [2]:
CONFIG = {
    "data_path":        "../preprocessing-dataset/processed_data.csv",
    "result_dir":       "result",
    "text_column":      "processed_text",
    "label_column":     "Label",
    "url_column":       "URL",
    "sample_size":      5000,
    "test_size":        0.2,
    "val_split":        0.2,
    "random_state":     42,
    "svd_components":   50,
    "n_pca_components": 4,
    "n_qubits":         4,
    "n_layers":         3,
    "n_epochs":         30,
    "lr":               0.01,
    "batch_size":       16,
}

os.makedirs(CONFIG["result_dir"], exist_ok=True)
print("Config loaded. Result dir ready.")

Config loaded. Result dir ready.


## 2. Load & Sample Data

Stratified 5 000-row sample from the full ~549 k dataset.

In [3]:
df_full = pd.read_csv(CONFIG["data_path"])
print(f"Full dataset shape: {df_full.shape}")

df, _ = train_test_split(
    df_full,
    train_size=CONFIG["sample_size"],
    random_state=CONFIG["random_state"],
    stratify=df_full[CONFIG["label_column"]],
)
df = df.reset_index(drop=True)

print(f"Sampled shape: {df.shape}")
print(f"Label distribution:\n{df[CONFIG['label_column']].value_counts()}")
df.head()

Full dataset shape: (549346, 3)
Sampled shape: (5000, 3)
Label distribution:
Label
good    3576
bad     1424
Name: count, dtype: int64


,URL,Label,processed_text
0,goldcoast.com.au/article/2011/11/02/362521_gol...,good,goldcoast com au articl 2011 11 02 362521 gold...
1,nypost.com/p/news/local/port_authority_boss_ch...,good,nypost com p news local port author boss chris...
2,yelp.ca/biz/redpath-sugar-museum-toronto,good,yelp ca biz redpath sugar museum toronto
3,theaircanadacentre.com/about/,good,theaircanadacentr com about
4,evri.com/organization/cinepop-0x2ff47,good,evri com organ cinepop 0x2ff47


## 3. URL Feature Extraction

6 handcrafted numeric features extracted before the train/test split.

In [4]:
def extract_url_features(url: str) -> list:
    """Extract 6 numeric features from a raw URL string.

    Args:
        url: Raw URL string.

    Returns:
        List of [url_length, num_dots, num_digits, num_special, has_ip, subdomain_depth].
    """
    return [
        len(url),
        url.count("."),
        sum(c.isdigit() for c in url),
        sum(c in "@-_=?&" for c in url),
        1 if re.search(r"\d+\.\d+\.\d+\.\d+", url) else 0,
        max(0, len(url.split("/")[2].split(".")) - 2) if "//" in url else 0,
    ]


url_features = np.array(
    [extract_url_features(str(u)) for u in df[CONFIG["url_column"]].fillna("")]
)
print(f"URL feature matrix shape: {url_features.shape}")

URL feature matrix shape: (5000, 6)


## 4. Train/Test Split

80/20 stratified. All sklearn transformers fitted on train only.

In [5]:
y          = df[CONFIG["label_column"]].values
text_col   = df[CONFIG["text_column"]].fillna("").astype(str).values

(
    text_train, text_test,
    url_feat_train, url_feat_test,
    y_train, y_test,
) = train_test_split(
    text_col, url_features, y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
    stratify=y,
)

print(f"Train: {len(text_train)}  |  Test: {len(text_test)}")

Train: 4000  |  Test: 1000


## 5. TF-IDF Vectorization

Fit on train only.

In [6]:
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=2, max_features=50000)
x_train_tfidf = tfidf.fit_transform(text_train)
x_test_tfidf  = tfidf.transform(text_test)

print(f"TF-IDF train: {x_train_tfidf.shape}  |  test: {x_test_tfidf.shape}")

TF-IDF train: (4000, 2303)  |  test: (1000, 2303)


## 6. TruncatedSVD — 50 Dense Dimensions

Fit on train only.

In [7]:
svd = TruncatedSVD(n_components=CONFIG["svd_components"], random_state=CONFIG["random_state"])
x_train_svd = svd.fit_transform(x_train_tfidf)
x_test_svd  = svd.transform(x_test_tfidf)

print(f"SVD explained variance : {svd.explained_variance_ratio_.sum():.2%}")

SVD explained variance : 33.85%


## 7. Combine Features + StandardScaler

hstack SVD (50) + URL features (6) → 56 dims. StandardScaler fit on train only.

In [8]:
x_train_combined = np.hstack([x_train_svd, url_feat_train])
x_test_combined  = np.hstack([x_test_svd,  url_feat_test])

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train_combined)
x_test_scaled  = scaler.transform(x_test_combined)

print(f"Combined + scaled train: {x_train_scaled.shape}")

Combined + scaled train: (4000, 56)


## 8. PCA → 4 Dimensions

One component per qubit. Fit on train only.

In [9]:
pca = PCA(n_components=CONFIG["n_pca_components"], random_state=CONFIG["random_state"])
x_train_pca = pca.fit_transform(x_train_scaled)
x_test_pca  = pca.transform(x_test_scaled)

print(f"PCA explained variance : {pca.explained_variance_ratio_.sum():.2%}")
print(f"PCA output shape       : {x_train_pca.shape}")

PCA explained variance : 12.76%
PCA output shape       : (4000, 4)


## 9. MinMaxScaler → [0, π]

Angle encoding requires values in [0, π]. Fit on train only.

In [10]:
angle_scaler   = MinMaxScaler(feature_range=(0, np.pi))
x_train_angles = angle_scaler.fit_transform(x_train_pca)
x_test_angles  = angle_scaler.transform(x_test_pca)

print(f"Angle train range: [{x_train_angles.min():.4f}, {x_train_angles.max():.4f}]")
print(f"Angle test range : [{x_test_angles.min():.4f}, {x_test_angles.max():.4f}]")
print(f"Shape            : {x_train_angles.shape}")

Angle train range: [0.0000, 3.1416]
Angle test range : [-0.0598, 3.1389]
Shape            : (4000, 4)


## 10. Label Encoding

`"bad" → 1.0`, `"good" → 0.0` for BCE loss.

In [11]:
label_map = {"bad": 1.0, "good": 0.0}

y_train_encoded = np.array([label_map[l] for l in y_train], dtype=np.float64)
y_test_encoded  = np.array([label_map[l] for l in y_test],  dtype=np.float64)

print(f"Train: bad={int(y_train_encoded.sum())}  good={int((1-y_train_encoded).sum())}")
print(f"Test : bad={int(y_test_encoded.sum())}   good={int((1-y_test_encoded).sum())}")

Train: bad=1139  good=2861
Test : bad=285   good=715


## 11. Train/Val Split

Split the 4 000-sample train set 80/20. The original 1 000-sample test set stays untouched until final evaluation.

In [12]:
x_tr, x_val, y_tr, y_val = train_test_split(
    x_train_angles,
    y_train_encoded,
    test_size=CONFIG["val_split"],
    random_state=CONFIG["random_state"],
    stratify=y_train_encoded,
)

print(f"VQC train : {len(x_tr)}  |  val : {len(x_val)}  |  test : {len(x_test_angles)}")

VQC train : 3200  |  val : 800  |  test : 1000


## 12. Quantum Device + VQC Circuit

**Interface:** `autograd` — PennyLane's native differentiable numpy backend (no PyTorch).  
**Gradient method:** `parameter-shift` — exact quantum gradients via the shift rule.  

Circuit per sample:  
1. `AngleEmbedding` — encode 4 input angles as RY rotations  
2. 3 variational layers: per-qubit RY + RZ, then CNOT ring (0→1→2→3→0)  
3. Measure Pauli-Z expectation on each qubit → 4 values in [−1, 1]

In [13]:
n_qubits: int = CONFIG["n_qubits"]
n_layers: int = CONFIG["n_layers"]

dev = qml.device("default.qubit", wires=n_qubits)


@qml.qnode(dev, interface="autograd", diff_method="parameter-shift")
def vqc_circuit(x: pnp.ndarray, params: pnp.ndarray) -> list:
    """VQC with angle encoding and parametric rotation + entanglement layers.

    Args:
        x: Input angles, shape (n_qubits,), values in [0, π].
        params: Rotation parameters, shape (n_layers, n_qubits, 2).

    Returns:
        List of n_qubits Pauli-Z expectation values in [−1, 1].
    """
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    for layer in range(n_layers):
        for i in range(n_qubits):
            qml.RY(params[layer, i, 0], wires=i)
            qml.RZ(params[layer, i, 1], wires=i)
        for i in range(n_qubits):
            qml.CNOT(wires=[i, (i + 1) % n_qubits])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


# Sanity check
dummy_x      = pnp.zeros(n_qubits, requires_grad=False)
dummy_params = pnp.zeros((n_layers, n_qubits, 2), requires_grad=False)
print("Circuit diagram:")
print(qml.draw(vqc_circuit)(dummy_x, dummy_params))
test_out = vqc_circuit(dummy_x, dummy_params)
print(f"\nOutput type : {type(test_out)}  length : {len(test_out)}")

Circuit diagram:
0: ─╭AngleEmbedding(M0)──RY(0.00)──RZ(0.00)─╭●───────╭X──RY(0.00)──RZ(0.00)─╭●───────╭X ···
1: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)─╰X─╭●────│───RY(0.00)──RZ(0.00)─╰X─╭●────│─ ···
2: ─├AngleEmbedding(M0)──RY(0.00)──RZ(0.00)────╰X─╭●─│───RY(0.00)──RZ(0.00)────╰X─╭●─│─ ···
3: ─╰AngleEmbedding(M0)──RY(0.00)──RZ(0.00)───────╰X─╰●──RY(0.00)──RZ(0.00)───────╰X─╰● ···

0: ··· ──RY(0.00)──RZ(0.00)─╭●───────╭X─┤  <Z>
1: ··· ──RY(0.00)──RZ(0.00)─╰X─╭●────│──┤  <Z>
2: ··· ──RY(0.00)──RZ(0.00)────╰X─╭●─│──┤  <Z>
3: ··· ──RY(0.00)──RZ(0.00)───────╰X─╰●─┤  <Z>

M0 = 
[0. 0. 0. 0.]

Output type : <class 'list'>  length : 4


## 13. Initialize Trainable Parameters & Optimizer State

All three parameter tensors are `pennylane.numpy` arrays with `requires_grad=True`.  
Adam momentum/velocity tracked as plain numpy arrays (no gradient needed for those).

In [14]:
np.random.seed(CONFIG["random_state"])

params            = pnp.array(np.random.randn(n_layers, n_qubits, 2) * 0.1, requires_grad=True)
classical_weights = pnp.array(np.random.randn(n_qubits) * 0.1,              requires_grad=True)
bias              = pnp.array(np.zeros(1),                                    requires_grad=True)

# Adam state (plain numpy — not differentiated)
_beta1, _beta2, _eps = 0.9, 0.999, 1e-8
_t = 0
_m_p  = np.zeros_like(params);            _v_p  = np.zeros_like(params)
_m_cw = np.zeros_like(classical_weights); _v_cw = np.zeros_like(classical_weights)
_m_b  = np.zeros_like(bias);              _v_b  = np.zeros_like(bias)


def adam_step(
    p: pnp.ndarray, g: np.ndarray,
    m: np.ndarray, v: np.ndarray,
    t: int, lr: float,
) -> tuple:
    """One Adam parameter update.

    Args:
        p: Current parameter array.
        g: Gradient array (same shape as p).
        m: First-moment estimate.
        v: Second-moment estimate.
        t: Current timestep (>= 1).
        lr: Learning rate.

    Returns:
        (new_param as pnp.array with requires_grad=True, new_m, new_v)
    """
    g_np  = np.array(g)
    m_new = _beta1 * m + (1 - _beta1) * g_np
    v_new = _beta2 * v + (1 - _beta2) * g_np ** 2
    m_hat = m_new / (1 - _beta1 ** t)
    v_hat = v_new / (1 - _beta2 ** t)
    p_new = np.array(p) - lr * m_hat / (np.sqrt(v_hat) + _eps)
    return pnp.array(p_new, requires_grad=True), m_new, v_new


n_vqc    = params.size
n_cls    = classical_weights.size + bias.size
print(f"VQC rotation params  : {n_vqc}  {list(params.shape)}")
print(f"Classical params     : {n_cls}")
print(f"Total trainable      : {n_vqc + n_cls}")

VQC rotation params  : 24  [3, 4, 2]
Classical params     : 5
Total trainable      : 29


## 14. Training Loop

Each batch:
1. Call `qml.grad(batch_cost, argnums=[0,1,2])` → parameter-shift gradients for VQC params + autograd for classical params  
2. Apply Adam update, re-wrap params as fresh `pnp.array(..., requires_grad=True)`  
3. Save best `params` (by val accuracy) to `result/vqc_weights.npy`

In [15]:
def batch_cost(
    p: pnp.ndarray,
    cw: pnp.ndarray,
    b: pnp.ndarray,
    bx: np.ndarray,
    by: np.ndarray,
) -> pnp.ndarray:
    """Mean BCE-with-logits loss over a mini-batch.

    Args:
        p:  VQC params, shape (n_layers, n_qubits, 2).
        cw: Classical weights, shape (n_qubits,).
        b:  Bias, shape (1,).
        bx: Batch features, shape (batch, n_qubits), plain numpy.
        by: Batch labels (0.0 / 1.0), plain numpy.

    Returns:
        Scalar mean loss.
    """
    total = pnp.array(0.0)
    for xi, yi in zip(bx, by):
        xi_arr = pnp.array(xi, requires_grad=False)
        out    = pnp.stack(vqc_circuit(xi_arr, p))          # (n_qubits,)
        logit  = pnp.dot(out, cw) + b[0]                    # scalar
        # numerically stable BCE: max(logit,0) - logit*y + log(1+exp(-|logit|))
        loss = (
            pnp.maximum(logit, pnp.array(0.0))
            - logit * float(yi)
            + pnp.log(pnp.array(1.0) + pnp.exp(-pnp.abs(logit)))
        )
        total = total + loss
    return total / float(len(bx))


def evaluate_val(
    xv: np.ndarray,
    yv: np.ndarray,
    p: pnp.ndarray,
    cw: pnp.ndarray,
    b: pnp.ndarray,
) -> float:
    """Compute validation accuracy (no gradient tracking).

    Args:
        xv: Validation features.
        yv: Validation labels.
        p, cw, b: Current parameters.

    Returns:
        Accuracy in [0, 1].
    """
    p_np  = np.array(p)
    cw_np = np.array(cw)
    b_np  = np.array(b)
    correct = 0
    for xi, yi in zip(xv, yv):
        out    = np.array(vqc_circuit(xi, p_np))
        logit  = float(np.dot(out, cw_np)) + float(b_np[0])
        pred   = 1 if 1.0 / (1.0 + np.exp(-logit)) >= 0.5 else 0
        correct += int(pred == int(yi))
    return correct / len(yv)


# -- gradient function (argnums 0,1,2 → params, classical_weights, bias)
grad_fn = qml.grad(batch_cost, argnums=[0, 1, 2])

train_losses:   list = []
val_accuracies: list = []
best_val_acc:   float = 0.0
best_epoch:     int   = 0
weights_path = os.path.join(CONFIG["result_dir"], "vqc_weights.npy")

n_train    = len(x_tr)
batch_size = CONFIG["batch_size"]
lr         = CONFIG["lr"]

for epoch in range(1, CONFIG["n_epochs"] + 1):
    perm       = np.random.permutation(n_train)
    x_shuf     = x_tr[perm]
    y_shuf     = y_tr[perm]
    epoch_loss = 0.0
    n_batches  = 0

    for start in range(0, n_train, batch_size):
        bx = x_shuf[start : start + batch_size]
        by = y_shuf[start : start + batch_size]

        # forward (cost value for logging)
        lv = float(batch_cost(np.array(params), np.array(classical_weights), np.array(bias), bx, by))

        # gradients via parameter-shift + autograd
        g_p, g_cw, g_b = grad_fn(params, classical_weights, bias, bx, by)

        # Adam updates
        _t += 1
        params,            _m_p,  _v_p  = adam_step(params,            g_p,  _m_p,  _v_p,  _t, lr)
        classical_weights, _m_cw, _v_cw = adam_step(classical_weights, g_cw, _m_cw, _v_cw, _t, lr)
        bias,              _m_b,  _v_b  = adam_step(bias,              g_b,  _m_b,  _v_b,  _t, lr)

        epoch_loss += lv
        n_batches  += 1

    avg_loss = epoch_loss / n_batches
    val_acc  = evaluate_val(x_val, y_val, params, classical_weights, bias)

    train_losses.append(avg_loss)
    val_accuracies.append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch   = epoch
        np.save(weights_path, np.array(params))

    print(f"Epoch {epoch:>2}/{CONFIG['n_epochs']} | Train Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

print(f"\nBest val acc: {best_val_acc:.4f} at epoch {best_epoch}")
print(f"Best VQC weights saved → {weights_path}")

Epoch  1/30 | Train Loss: 0.5589 | Val Acc: 0.7450
Epoch  2/30 | Train Loss: 0.5170 | Val Acc: 0.7650


KeyboardInterrupt: 

## 15. Plot Training Curve

Dual-axis: train loss (left, blue) and val accuracy (right, orange). Dashed line marks the best epoch.

In [ ]:
epochs_range = list(range(1, CONFIG["n_epochs"] + 1))

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train Loss", color="steelblue")
ax1.plot(epochs_range, train_losses, color="steelblue", linewidth=2, label="Train Loss")
ax1.tick_params(axis="y", labelcolor="steelblue")

ax2 = ax1.twinx()
ax2.set_ylabel("Val Accuracy", color="darkorange")
ax2.plot(epochs_range, val_accuracies, color="darkorange", linewidth=2, label="Val Accuracy")
ax2.tick_params(axis="y", labelcolor="darkorange")
ax2.set_ylim(0.0, 1.0)

ax1.axvline(x=best_epoch, color="gray", linestyle="--", linewidth=1.5, label=f"Best epoch ({best_epoch})")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.title("VQC Training Curve")
plt.tight_layout()

curve_path = os.path.join(CONFIG["result_dir"], "training_curve_vqc.png")
fig.savefig(curve_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Training curve saved → {curve_path}")

## 16. Load Best Weights & Evaluate on Test Set

Reload the checkpoint saved at the best validation epoch, then run inference on the held-out test set.

In [ ]:
def predict(
    x_batch: np.ndarray,
    p: pnp.ndarray,
    cw: pnp.ndarray,
    b: pnp.ndarray,
) -> np.ndarray:
    """Run inference on a batch without gradient tracking.

    Args:
        x_batch: Features, shape (n, n_qubits).
        p: VQC params.
        cw: Classical weights.
        b: Bias.

    Returns:
        Integer predictions array (0 or 1).
    """
    p_np  = np.array(p)
    cw_np = np.array(cw)
    b_np  = np.array(b)
    preds = []
    for xi in x_batch:
        out   = np.array(vqc_circuit(xi, p_np))
        logit = float(np.dot(out, cw_np)) + float(b_np[0])
        preds.append(1 if 1.0 / (1.0 + np.exp(-logit)) >= 0.5 else 0)
    return np.array(preds)


# Reload best VQC weights
best_params_np = np.load(weights_path)
params         = pnp.array(best_params_np, requires_grad=True)
print(f"Best weights loaded from {weights_path}  (epoch {best_epoch})")

# Predict on test set
y_pred_int    = predict(x_test_angles, params, classical_weights, bias)
inv_label_map = {1: "bad", 0: "good"}
y_pred_labels = np.array([inv_label_map[p] for p in y_pred_int])
y_test_labels = y_test  # original string labels

test_acc = accuracy_score(y_test_labels, y_pred_labels)
report   = classification_report(y_test_labels, y_pred_labels, labels=["bad", "good"])

print(f"\nTest Accuracy: {test_acc:.4f}\n")
print(report)

## 17. Plot & Save Confusion Matrix

In [ ]:
cm     = confusion_matrix(y_test_labels, y_pred_labels, labels=["bad", "good"])
labels = ["bad", "good"]

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title("VQC Confusion Matrix")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()

cm_path = os.path.join(CONFIG["result_dir"], "confusion_matrix_vqc.png")
fig.savefig(cm_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Confusion matrix saved → {cm_path}")

## 18. Save Classification Report & Confirm Outputs

In [ ]:
report_path = os.path.join(CONFIG["result_dir"], "classification_report_vqc.txt")
with open(report_path, "w") as f:
    f.write(f"Test Accuracy: {test_acc:.4f}\n\n")
    f.write(report)

print(f"classification_report_vqc.txt → {report_path}")
print(f"confusion_matrix_vqc.png      → {cm_path}")
print(f"training_curve_vqc.png        → {curve_path}")
print(f"vqc_weights.npy               → {weights_path}")
print(f"Saved weights shape           : {np.load(weights_path).shape}")

## 19. Summary

In [ ]:
print("=" * 60)
print("  VQC — Phishing URL Detection — Final Summary")
print("=" * 60)
print("\n[Config]")
for key, val in CONFIG.items():
    print(f"  {key:<22}: {val}")
print("\n[Pipeline]")
print(f"  SVD explained variance   : {svd.explained_variance_ratio_.sum():.2%}")
print(f"  PCA explained variance   : {pca.explained_variance_ratio_.sum():.2%}")
print(f"  Angle encoding range     : [0, π]")
print("\n[VQC Architecture]")
print(f"  n_qubits                 : {n_qubits}")
print(f"  n_layers                 : {n_layers}")
print(f"  Encoding                 : AngleEmbedding (RY)")
print(f"  Entanglement             : CNOT ring")
print(f"  Gradient method          : parameter-shift (PennyLane autograd)")
print(f"  VQC rotation params      : {params.size}")
print(f"  Classical params         : {classical_weights.size + bias.size}")
print(f"  Total trainable          : {params.size + classical_weights.size + bias.size}")
print("\n[Training]")
print(f"  Best epoch               : {best_epoch}/{CONFIG['n_epochs']}")
print(f"  Best val accuracy        : {best_val_acc:.4f}")
print("\n[Results]")
print(f"  Test accuracy            : {test_acc:.4f}")
print("\n[Baselines for comparison]")
baselines = {
    "KNN":                  0.9186,
    "Logistic Regression":  0.9249,
    "Naive Bayes":          0.9698,
    "SVM RBF":              0.9037,
    "Random Forest":        0.9166,
    "MLP NN":               0.9232,
    "QKNN":                 0.7780,
    "VQC (this model)":     test_acc,
}
for name, acc in baselines.items():
    marker = " ◄" if name == "VQC (this model)" else ""
    print(f"  {name:<22}: {acc:.4f}{marker}")
print("=" * 60)